## CNN : 이미지에서 패턴을 찾는 기계
1. 이미지 → 특징 벡터
2. ConvNeXt 뒤에 작은 신경망을 붙인다
3. 모델의 학습 방식 => 사진 → 예측 → 틀림 → 수정의 반복
- ConvNeXt는 큰 영역 패턴을 잘 본다.

## ConvNeXt Tiny?
- ConvNeXt에는 여러 크기가 있지만
- 속도 빠름, 과적합 적음, GPU 부담 적음

### 이 모델의 약점
- 카메라를 아래로 기울이면 "경사구나", 카메라 자세가 영향을 줌
- 그래서 우리가 해야 할것?
  * ROI crop
  * augmentation

### MAE (Masked AutoEncoder)
이미지 일부를 가린다
↓
모델이 복원하도록 학습

In [40]:
import torch
torch.cuda.is_available()

True

In [41]:
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset
import numpy as np
import time
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import os
import re
import math
import numpy as np
import pandas as pd
import cv2
from configparser import ConfigParser
from glob import glob # 파일 경로를 패턴으로 검색해 리스트로 가져오는 함수

from PIL import Image

In [42]:
class WheelSafeSlopeDataset(Dataset):
    def __init__(self, dataFrame, transform=None):
        self.df = dataFrame.reset_index(drop=True).copy()
        self.transform = transform
        self.df = self.df[~self.df["path"].astype(str).str.contains("_debug_", na=False)].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "path"]
        label = self.df.loc[idx, "slope_avg"]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = torch.tensor(label, dtype=torch.float32)
        return img, label

## transform 설계
1. 바닥 경사는 화면 하단 영역 정보가 핵심
  * 휠체어 위험은 대부분 **가까운 바닥(화면 아래)**에서 결정됨.
  * 따라서 Bottom-biased crop이 가장 큰 성능 상승 요인 중 하나입니다.

2. 라벨(각도)을 망가뜨리는 증강은 금지
  * 이미지를 크게 회전(rotate)하면 경사 방향이 바뀌거나, 카메라 roll/pitch를 비현실적으로 바꾸게 되어 라벨과 불일치가 발생할 수 있음.
  * 그래서 **회전은 아주 소량(roll 흉내)**만 허용하고, pitch는 "적당한 perpective jitter"로만

3. 스마트폰 환경을 흉내
  * 노출 변화(밝기/그늘), 컬러 밸런스, 노이즈, 약한 블러는 현실에서 흔함 -> 학습에 반드시 넣기

4. ConvNeXt는 224보다 384에서 "장면 기하 힌트"를 더 잘 먹는 편
  * 경사는 미세한 원근 단서가 중요해서, 가능하면 384 입력을 추천합니다. (성능 vs 속도 trade off)

In [43]:
import random
from PIL import Image

class BottomBiasedCrop:
    """
    화면 하단 위주로 crop해서 '가까운 바닥(주행면)' 정보에 집중하게 함.
    - crop_h_ratio: crop 높이 비율 범위 (예: 0.55~0.75)
    - crop_w_ratio: crop 너비 비율 범위 (예: 0.80~1.00)
    - bottom_bias: crop의 top 좌표를 아래로 더 치우치게 하는 정도(0~1)
    """
    def __init__(self, crop_h_ratio=(0.55, 0.75), crop_w_ratio=(0.80, 1.00), bottom_bias=0.85):
        self.crop_h_ratio = crop_h_ratio
        self.crop_w_ratio = crop_w_ratio
        self.bottom_bias = bottom_bias

    def __call__(self, img: Image.Image) -> Image.Image:
        W, H = img.size
        ch = int(H * random.uniform(*self.crop_h_ratio))
        cw = int(W * random.uniform(*self.crop_w_ratio))

        # x는 중앙 근처에서 조금 흔들기
        x0_max = max(0, W - cw)
        x0 = int(random.uniform(0, x0_max))

        # y는 하단 편향: 가능한 범위의 아래쪽을 더 자주 선택
        y0_max = max(0, H - ch)
        # bottom_bias가 클수록 아래쪽(y0가 큼)을 더 자주 뽑음
        u = random.random() ** (1.0 / max(1e-6, self.bottom_bias))
        y0 = int(u * y0_max)

        return img.crop((x0, y0, x0 + cw, y0 + ch))

In [44]:
class BottomCenterCrop:
    """
    평가용 고정 하단 crop
    - 화면 하단 중심부를 deterministic 하게 crop
    """
    def __init__(self, crop_h_ratio=0.65, crop_w_ratio=0.90):
        self.crop_h_ratio = crop_h_ratio
        self.crop_w_ratio = crop_w_ratio

    def __call__(self, img: Image.Image) -> Image.Image:
        W, H = img.size
        ch = int(H * self.crop_h_ratio)
        cw = int(W * self.crop_w_ratio)

        x0 = max(0, (W - cw) // 2)   # 가로 중앙
        y0 = max(0, H - ch)          # 세로 하단 고정

        return img.crop((x0, y0, x0 + cw, y0 + ch))

In [45]:
from torchvision import transforms

IMG_SIZE = 384  # 224도 가능하지만, 경사 단서 보존을 위해 384 추천

train_transform = transforms.Compose([
    BottomBiasedCrop(crop_h_ratio=(0.55, 0.75), crop_w_ratio=(0.80, 1.00), bottom_bias=0.85),     # 1) 바닥 중심 crop (도메인 핵심)
    transforms.Resize((IMG_SIZE, IMG_SIZE)),     # 2) 촬영 거리/구도 변화 흉내: resize로 표준화
    transforms.RandomApply([transforms.RandomRotation(degrees=3)], p=0.4), # 3) 카메라 roll 오차(좌우 기울기)만 아주 소량 허용 (큰 회전은 라벨을 깨므로 금지)
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.03), # 4) 스마트폰 환경(조명/색) 변화
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.2))], p=0.2), # 5) 약한 블러/노이즈 (현실 촬영 대비)
    transforms.ToTensor(), # 6) 텐서화
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # 7) 정규화
])

val_transform = transforms.Compose([
    # 평가에서는 랜덤성 제거. 다만 휠체어 서비스 목적상 하단 정보 집중은 유지 가능.
    # (만약 "전체 화면" 기준으로 평가하고 싶으면 BottomBiasedCrop 제거하고 CenterCrop 사용)
    BottomCenterCrop(crop_h_ratio=0.65, crop_w_ratio=0.90),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

### DataSet 클래스 완성
- WheelSafeSlopeDataset 유지

In [ ]:
from lib.utils.path import output_path

# 1) CSV 로드
csv_path = output_path() / 'slope_labels_20260309_185927.csv'
df = pd.read_csv(csv_path)

df["slope_avg"] = pd.to_numeric(df["slope_avg"], errors="coerce")
df = df[df['slope_avg'] < 17]
print(len(df))

# 2) train/val/test split (재현성을 위해 random_state 고정)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df  = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)

print(len(train_df), len(val_df), len(test_df))

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# 3) Dataset 생성
train_dataset = WheelSafeSlopeDataset(train_df, transform=train_transform)
val_dataset   = WheelSafeSlopeDataset(val_df,   transform=val_transform)
test_dataset  = WheelSafeSlopeDataset(test_df,  transform=val_transform)

TypeError: bad operand type for abs(): 'str'

## DataLoader 생성
- batch_size (GPU 있으면 16~64, 없으면 4~16)
- num_workers (윈도우면 0~4부터)
- pin_memory=True(GPU)
- shuffle=True(train), shuffle=False(val/test)

### DataLoader가 하는 일
- PyTorch 학습은 매 Step마다 대략 이렇게 돕니다.
1. (CPU) 디스크에서 이미지 읽기 + 전처리(transform)
2. (CPU) 배치로 묶기(collate)
3. (CPU→GPU) 배치 텐서를 GPU로 복사
4. (GPU) forward/backword
- 여기서 병목이 자주 생기는 곳이 1~3번 (데이터 공급) 입니다. DataLoader는 이 구간을 최적화하기 위한 도구예요

### 옵션 설명
1. `batch_size`
- 한 번에 몇 장을 묶어서 학습할지
- 커질수록
  * GPU 활용률 ↑ (빠를 수 있음)
  * 메모리 사용량 ↑
  * 통계쩍으로 더 안정적인 gradient
- 작아질수록:
  * 메모리 절약
  * 대신 noisy gradient로 학습이 흔들릴 수 있음
- 권장 시작값
  * GPU 8~12GB: batch_size=16부터
  * CPU만: batch_size=4~8

2. `num_worker`
- 데이터를 로드/전처리 하는 "백그라운드 워크 프로세스" 수
- 늘릴수록:
  * CPU가 병렬로 

In [ ]:
# 4) DataLoader 생성 (윈도우 기준 안전 시작 세팅)
BATCH_SIZE = 4  # GPU 있으면 16부터, 없으면 4~8
NUM_WORKERS = 0  # Windows는 0부터 시작해서 안정 확인 후 2,4로 올리기
PIN_MEMORY = True  # GPU 없으면 False로

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# 5) 동작 확인: 한 배치 꺼내보기
imgs, labels = next(iter(train_loader))
print("batch imgs:", imgs.shape, imgs.dtype)
print("batch labels:", labels.shape, labels.dtype)

batch imgs: torch.Size([4, 3, 384, 384]) torch.float32
batch labels: torch.Size([4]) torch.float32


### Model 단계
- Backbone 로딩 (timm 추천)
 * convnextv2_tiny 또는 convnextv2_small
 * pretrained 사용
 * 마지막 classifier 제거 후 1-d regression head 부착

- backbone : 이미지의 이해(이미지 특징 추출기)
- Head : 원하는 출력 만들기
- pretrained : 이미 엄청 많은 이미지로 학습된 모델입니다.
  * 즉 모델이 이미 이런 것들을 배웠습니다. edge / texture / geometry / object structure / perspective

### 구조의 동작
Image
 ↓
ConvNeXtV2 (Backbone)
 ↓
Feature Vector
 ↓
Regression Head
 ↓
Slope Angle

### classifier 제거 후 regression head 부착
- 매우 중요한 부분
- ConvNeXtV2는 원래 **분류 모델** 입니다.
  * Linear(768 → 1000) 여기서 1000은 ImageNet class 수
- 그래서 classifier를 제거한다.
ConvNeXt
 ↓
Linear(768 → 1000)
 ↓
class
우리는 이걸 제거 하고
ConvNeXt
 ↓
Linear(768 → 1)
 ↓
angle

### Regression head가 무엇인가?
* 1-d = 출력 차원이 1개라는 뜻
* Regression head = 분류 확률이 아니라 연속값을 내는 출력층

### Transfer Learning vs Fine-tuning
(1) Transfer Learning
- 이미 학습된 모델을 가져와서 다른 문제에 활용하는 것

(2) Fine-tuning
- 새 데이터에 맞게 추가로 학습

In [ ]:
import torch
import torch.nn as nn
import timm

class ConvNeXtV2Regressor(nn.Module):
    def __init__(self, backbone_name="convnextv2_tiny", pretrained=True, dropout=0.1):
        super().__init__()

        # 1) timm에서 backbone 로드
        # num_classes=0 => classifier 제거 + feature만 뽑도록 설정
        # global_pool='avg' => Global Average Pooling 적용된 feature 반환
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0, # 마지막에 num_classes개 출력(예: 1000개)을 만드는 classifier가 붙어 있는데, num_classes=0으로 주면 **그 classifier를 제거하고 “feature만 출력”**하게 됩니다.
            global_pool="avg" # ? 이거 뭔지 모르겠음
        )

        # 2) backbone이 뽑는 feature 차원 확인
        feat_dim = self.backbone.num_features # 예를 들어 tiny면 대략 768 같은 값이 나올 수 있습니다. 이후 head의 입력 차원을 맞추기 위해 필요합니다.

        # 3) 1-d regression head 부착
        # (feature -> angle(1개))
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim), # feature 벡터를 정규화(Layer Normalization)합니다
            nn.Dropout(dropout), # 학습 중 일부 feature를 랜덤으로 꺼서(0으로 만들어서) 과적합을 줄입니다. (dropout=0.1이면 10% 정도를 랜덤으로 끕니다.)
            nn.Linear(feat_dim, 1) # **회귀 출력층** (feature 벡터([feat_dim])를 입력 받아서 **숫자 1개(각도 1개)**를 출력합니다.)
        )

    # (입력 → 출력 흐름)
    def forward(self, x):
        feat = self.backbone(x)      # backbone에 이미지를 넣어 feature 벡터를 얻습니다. [B, feat_dim]
        out = self.head(feat)        # feature를 head에 넣어 각도를 예측합니다.[B, 1]
        return out.squeeze(1)        # 결과는 [B, 1] (배치마다 숫자 1개). [B]

# ---- 동작 확인 ----
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu" # GPU가 있으면 cuda, 없으면 cpu를 사용합니다.

    model = ConvNeXtV2Regressor(
        backbone_name="convnextv2_tiny",  # 또는 "convnextv2_small"
        pretrained=True,
        dropout=0.1
    ).to(device)

    x = torch.randn(4, 3, 384, 384).to(device)  # 더미 입력을 만듭니다. 배치 4장, RGB 3채널, 384x384 크기의 랜덤 이미지.
    y = model(x) # 모델에 입력을 넣고 예측값을 얻습니다.

    print("output shape:", y.shape)  # torch.Size([4])
    print("example:", y[:2])

output shape: torch.Size([4])
example: tensor([ 0.3096, -0.2595], device='cuda:0', grad_fn=<SliceBackward0>)


5) Loss / Optimizer / Scheduler
- Loss: HuberLoss(SmoothL1Loss) 권장
- Optimizer: AdamW
- Scheduler: CosineAnnealingLR 또는 ReduceLROnPlateau
- (권장) AMP + grad clipping

1. 왜 Huber(SmoothL1Loss)인가?
- stereo disparity 노이즈
- confidence 마스크 품질 편차
- ground ROI가 계단/턱/차도 경계 등을 포함
- plane fitting(RANSAC) 실패/부분 실패

이럴 때 **MSE(L2)**는 이상치에 크게 끌려가서 학습이 흔들립니다.
**Huber(SmoothL1)**는 “작은 오차는 L2처럼 부드럽게, 큰 오차는 L1처럼 완만하게” 처리해서 더 안정적입니다.

작은 오차 구간: 빠르게 수렴(L2 장점)
큰 오차 구간: 이상치 영향 완화(L1 장점)

PyTorch에서는 nn.SmoothL1Loss가 Huber loss입니다.

2. Optimizer: 왜 AdamW인가?
- Adam vs AdamW
- Adam: weight decay가 “L2 정규화”처럼 동작하지 않는 이슈가 있음
- AdamW: weight decay를 **파라미터 업데이트에서 분리(decoupled)**하여 일반화가 더 안정적인 경우가 많음
- 특히 pretrained backbone fine-tuning에서 AdamW가 매우 흔한 선택입니다.

3. Scheduler: CosineAnnealingLR vs ReduceLROnPlateau
(A) CosineAnnealingLR (추천 기본)

학습률을 코사인 곡선 형태로 서서히 줄임

“딱히 val loss를 보고 조절하지 않아도” 잘 동작

설정 쉽고 안정적

(B) ReduceLROnPlateau (데이터 작을 때 유리한 경우)

val loss가 안 좋아지면 lr을 줄임

다만 “val metric이 조금 흔들리는” 상황에선 너무 빨리 줄어들 수 있음

권장

처음은 CosineAnnealingLR로 깔끔하게 가고

과적합/진동이 심하면 ReduceLROnPlateau로 전환

4) AMP (mixed precision)와 grad clipping
AMP

float16/float32 혼합 사용 → 속도↑ / 메모리↓

GPU(CUDA) 환경에서 특히 유리

grad clipping

큰 gradient 폭발을 방지

fine-tuning에서 “갑자기 loss가 튀는” 현상 완화에 도움

### Optimizer(옵티마이저)
> 모델의 가중치(weight)를 어떻게 업데이트할지 결정하는 알고리즘
- 모델의 틀린 정도(loss)를 보고 가중치를 얼마나 / 어떤 방향으로 바꿀지 계산하는 역할

#### 딥러닝의 학습의 순서
Input
  ↓
Model
  ↓
Prediction
  ↓
Loss 계산
  ↓
Gradient 계산 (backpropagation)
  ↓
Optimizer가 weight 업데이트
* 여기서 optimizer가 실제로 모델을 학습시키는 역할입니다.

### Optimizer가 하는 일 (수식 느낌)
- 가장 기본적인 방식은 Gradient Descent입니다.
* w_new = w_old(모델 가중치) - learning_rate(얼마나 크게 이동할지) * gradient(loss의 기울기)
> loss가 줄어드는 방향으로 weight를 이동시킵니다.

### Adam 이란?
> Adaptive Moment Estimation
gradient 평균 + gradient 분산을 동시에 사용합니다.
즉 gradient 크기에 따라 learning rate를 자동 조절합니다.
- AdamW? weight decay를 분리(decoupled) 그래서 generalization 성능 ↑
#### weight decay?
- 모델이 너무 복잡해지는 것을 막는 정규화 방법
### Optimizer가 코드에서 하는 일
optimizer.zero_grad()   # gradient 초기화

loss.backward()         # gradient 계산

optimizer.step()        # weight 업데이트

### Optuna?
> Hyperparameter Optimization Library
모델 학습 파라미터를 자동으로 탐색합니다.

learning_rate
batch_size
dropout
optimizer 종류
등

를 자동으로 찾습니다.

### Scheduler?
> 학습 중 Learning Rate(LR)를 자동으로 조절하는 알고리즘
 * optimizer = "어떻게 weight를 업데이트할지"
 * scheduler = "learning rate를 언제/얼마나 바꿀지"
#### 왜 바꾸는데?
딥러닝에서는 보통
- 초반 → LR 크게 (빠르게 탐색)
- 후반 → LR 작게 (정밀 조정)
이 전략이 가장 좋다.
Scheduler는 바로 이 작업을 한다.

#### Optimizer vs Scheduler
- Optimizer | gradient로 weight 업데이트
- Sceduler | learning rate 변화

#### CosineAnnealingLR 원리
- learning rate를 cosine 곡선 형태로 감소시킴
LR
│\
│ \
│  \
│   \__
│      \__
└──────────── epoch
- lr = eta_min + (lr_max - eta_min) * (1 + cos(pi * t / T)) / 2
- 초반 큰 LR > 중반 천천히 감소 > 후반 아주 작은 LR (fine tuning 성능이 좋음)
#### ReduceLROnPlateau 원리
- validation loss가 좋아지지 않으면 **learning rate를 줄인다**

## Epoch?
- 전체 training dataset을 모델이 한 번 모두 학습한 것
### Epoch을 여러 번 하는 이유?
- 처음에는 모델이 랜덤 상태 > 여러 번 데이터셋을 반복 학습해야 합니다.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -------------------------
# 0) Device
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda") # GPU 있으면 GPU로 학습하고, GPU일 때만 AMP를 켜는 설정

# -------------------------
# 1) Loss (Huber / SmoothL1)
# 회귀 손실함수 생성기를 정의
# -------------------------
def build_regression_loss(beta: float = 1.0) -> nn.Module:
    """
    SmoothL1Loss == HuberLoss (PyTorch 버전에 따라 표기 차이)
    - beta가 작을수록 L1에 가까워져(outlier에 더 robust)
    - beta가 클수록 L2에 가까워짐(미세 오차에 민감)
    """
    # PyTorch >= 1.10: SmoothL1Loss(beta=...)
    try:
        return nn.SmoothL1Loss(beta=beta) # 최신 PyTorch면 beta를 지정해 SmoothL1Loss를 만든다
    except TypeError:
        return nn.SmoothL1Loss() # 구버전 호환: SmoothL1Loss에 beta가 없으면 기본 설정 사용

criterion = build_regression_loss(beta=1.0)

# -------------------------
# 2) Optimizer (AdamW) + param groups (no weight decay for norm/bias)
# -------------------------
def build_adamw_optimizer(model: nn.Module, lr: float = 3e-4, weight_decay: float = 1e-2):
    """
    일반적으로 bias와 LayerNorm 계열 파라미터에는 weight_decay를 적용하지 않는 게 안정적.
    ConvNeXt는 LayerNorm을 많이 사용하므로 특히 효과적입니다.
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # bias 또는 normalization 계열 파라미터는 no_decay로 분리
        if name.endswith(".bias") or ("norm" in name.lower()) or ("ln" in name.lower()):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    param_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    optimizer = optim.AdamW(param_groups, lr=lr, betas=(0.9, 0.999))
    return optimizer

# 예시: Tiny 기준 시작값
# (Small이면 lr을 2e-4 정도로 낮추는 것도 흔한 선택)
optimizer = build_adamw_optimizer(model, lr=3e-4, weight_decay=1e-2)

# -------------------------
# 3) Scheduler
# -------------------------
def build_scheduler(optimizer, scheduler_type: str, num_epochs: int):
    """
    scheduler_type:
      - "cosine": CosineAnnealingLR (epoch 수가 정해진 경우 추천)
      - "plateau": ReduceLROnPlateau (val loss 기반)
    """
    scheduler_type = scheduler_type.lower() # 타입을 소문자로 변환(COSINE, Cosine, cosine등에 대응)

    if scheduler_type == "cosine":
        # T_max: 한 주기(보통 전체 epoch) / eta_min: 최저 lr
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    if scheduler_type == "plateau":
        # val_loss가 개선 없으면 lr 감소
        return optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=5,
            threshold=1e-4,
            min_lr=1e-6,
        )

    raise ValueError(f"Unknown scheduler_type: {scheduler_type}")

NUM_EPOCHS = 10
SCHEDULER_TYPE = "cosine"   # 또는 "plateau"
scheduler = build_scheduler(optimizer, SCHEDULER_TYPE, NUM_EPOCHS)

# -------------------------
# 4) AMP scaler (GPU일 때만)
# -------------------------
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# -------------------------
# 5) Grad clipping 설정
# -------------------------
MAX_GRAD_NORM = 1.0

# 가끔 gradient가 매우 커지는 경우가 있음. > gradient가 일정 크기 이상 커지지 않도록 제한하는 기법
# max_norm: gradient 크기 제한
# norm : 벡터의 크기
# Gradient clipping은 반드시 backward 이후에 사용
def clip_gradients(model: nn.Module, max_norm: float = 1.0):
    """
    training loop에서 backward 후 optimizer.step 전에 호출.
    """
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)

print("Device:", device)
print("AMP enabled:", use_amp)
print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Scheduler:", scheduler.__class__.__name__)

Device: cuda
AMP enabled: True
Loss: SmoothL1Loss()
Optimizer: AdamW
Scheduler: CosineAnnealingLR


C:\Users\user\AppData\Local\Temp\ipykernel_45652\2112217588.py:97: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


6) Training (여기서부터 “train data 학습”)

Train loop

model.train()

배치마다: forward → loss → backward → optimizer.step()

Validation loop

model.eval() + torch.no_grad()

val loss/MAE 계산

Checkpoint 저장

best_val_mae(또는 best_val_loss) 갱신 시 저장

(선택) Early stopping

## tqdm  - 학습 진행 상황 표시기
- tqdm은 progress bar 라이브러리.

## Summary Writer - 학습 로그 기록기
- TesorBoard 로그를 기록하는 도구
> TensorBoard: 딥러닝 학습을 그래프로 시각화하는 도구

## AMP
### Precision이란?
- 딥러닝은 보통 float32를 사용한다.
- 하지만 GPU는 float16 연산이 훨씬 빠르다.
- float16은 정밀도 부족 문제가 있다. (gradient = 0.00000012가 0으로 사라질 수 있다. underflow라고 한다.)
- AMP는 일부 연산 → float16, 중요 연산 → float32

### AMP 내부 동작 AMP는 두 가지 핵심 기술을 사용한다.
1. autocast (연산을 자동으로 fp16 / fp32 선택)
2. GradScaler (fp16 gradient underflow > loss를 크게 만든 후 gradient 계산)

In [ ]:
import os
import time
import torch
import numpy as np

from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

# [ADDED] Optuna 추가
import optuna
from optuna.pruners import MedianPruner


# -------------------------
# Metrics
# -------------------------

# MAE 계산 함수(Mean Absolute Error_평균 절대 오차)
def mae_deg(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    pred/target shape: (B,) or (B,1)
    return: scalar tensor
    """
    # (B,) 형태로 1차원 벡터로 변환
    # 예) (B,1) → (B,)
    pred = pred.view(-1)
    target = target.view(-1)

    # MAE 계산
    return torch.mean(torch.abs(pred - target))


# -------------------------
# One epoch train
# train dataset을 1번 전부 학습하는 함수
# 한 epoch은 for batch in train_loader 형태로 진행 됩니다.
# dataset = 1000 images batch = 20 → 50 step
# -------------------------
def train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
    scaler=None,
    use_amp: bool = True,
    max_grad_norm: float = 1.0,
    log_interval: int = 50,
    writer=None,         # [ADDED] TensorBoard writer
    epoch=None,          # [ADDED] 현재 epoch 번호 기록용
):
    model.train() # 모델을 학습 모드로 전환

    # 로그 변수 초기화
    running_loss = 0.0
    running_mae = 0.0
    n_samples = 0

    start = time.time() # epoch 시간 측정

    # [ADDED] tqdm progress bar 적용
    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch}", leave=False)

    # 핵심 루프
    # train_loader가 반환하는 것 (imgs, labels)
    # imgs shape = (B, 3, 384, 384)
    # labels shape = (B,)
    for step, (imgs, labels) in enumerate(train_loader):

        # CPU → GPU 이동
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().view(-1)  # (B,)

        # gradient 초기화(딥러닝에서 gradient는 누적된다.)
        optimizer.zero_grad(set_to_none=True)

        # AMP 학습 분기(AMP 사용시 mixed precision 학습 진행)
        if use_amp and scaler is not None:
            # autocast (이 블ㄹ록 내부에서 FP16 + FP32 자동 혼합)
            with torch.cuda.amp.autocast():
                preds = model(imgs).view(-1)  # (B,)
                loss = criterion(preds, labels)

            scaler.scale(loss).backward()

            # grad clipping은 unscale 후
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(imgs).view(-1)
            loss = criterion(preds, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()

        batch_size = imgs.size(0)
        running_loss += loss.item() * batch_size
        running_mae += mae_deg(preds.detach(), labels).item() * batch_size
        n_samples += batch_size

        # [ADDED] tqdm postfix 표시
        avg_loss = running_loss / max(1, n_samples)
        avg_mae = running_mae / max(1, n_samples)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", mae=f"{avg_mae:.4f}")

        # [ADDED] step 단위 TensorBoard 기록
        if writer is not None:
            global_step = (epoch - 1) * len(train_loader) + step
            writer.add_scalar("Step/train_loss", loss.item(), global_step)
            writer.add_scalar("Step/train_mae", mae_deg(preds.detach(), labels).item(), global_step)


        if (step + 1) % log_interval == 0:
            avg_loss = running_loss / n_samples
            avg_mae = running_mae / n_samples
            print(f"[Train] step {step+1}/{len(train_loader)} | loss {avg_loss:.4f} | mae {avg_mae:.4f}")

    epoch_loss = running_loss / max(1, n_samples)
    epoch_mae = running_mae / max(1, n_samples)
    elapsed = time.time() - start
    return epoch_loss, epoch_mae, elapsed


# -------------------------
# Validation
# -------------------------

# Validation Loop (검증 단계) Train loop와 매우 비슷하지만 **“학습을 하지 않는다”**는 점이 핵심 차이
# validation dataset 전체를 한번 평가(val dataset → 모델 예측 → loss / mae 계산 → 평균 반환)
# @torch.no_grad() "이 함수에서는 gradient 계산하지 마라"
@torch.no_grad()
def validate(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_mae = 0.0
    n_samples = 0

    # [ADDED] tqdm progress bar 적용
    pbar = tqdm(val_loader, desc="Validation", leave=False)

    for imgs, labels in pbar:
        # GPU 이동
        imgs = imgs.to(device, non_blocking=True)

        # label shape 정리
        labels = labels.to(device, non_blocking=True).float().view(-1)

        # Forward (예측) 모델이 image → slope angle를 예측
        preds = model(imgs).view(-1)

        # Loss 계산 (전체 dataset 평균을 구하기 위해 곱합니다.)
        loss = criterion(preds, labels)

        # batch size 계산
        batch_size = imgs.size(0)

        # loss 누적
        running_loss += loss.item() * batch_size

        # MAE 누적(|pred - label|)
        running_mae += mae_deg(preds, labels).item() * batch_size
        n_samples += batch_size

        # [ADDED] tqdm postfix 표시
        avg_loss = running_loss / max(1, n_samples)
        avg_mae = running_mae / max(1, n_samples)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", mae=f"{avg_mae:.4f}")
    
    # 평균 계산
    val_loss = running_loss / max(1, n_samples)
    val_mae = running_mae / max(1, n_samples)
    return val_loss, val_mae


# -------------------------
# Checkpoint helpers
# -------------------------
def save_checkpoint(path, model, optimizer, epoch, best_val_mae, scheduler=None, scaler=None):
    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "best_val_mae": best_val_mae,
    }
    if scheduler is not None:
        ckpt["scheduler"] = scheduler.state_dict()
    if scaler is not None:
        ckpt["scaler"] = scaler.state_dict()

    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(ckpt, path)


# -------------------------
# Early stopping (optional)
# -------------------------
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.num_bad = 0

    def step(self, metric_value: float) -> bool:
        """
        return True if should stop
        """
        if self.best is None or metric_value < (self.best - self.min_delta):
            self.best = metric_value
            self.num_bad = 0
            return False
        else:
            self.num_bad += 1
            return self.num_bad >= self.patience


# -------------------------
# Main training loop
# -------------------------
def fit(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    num_epochs: int = 10,
    scheduler_type: str = "cosine",  # "cosine" or "plateau"
    use_amp: bool = True,
    scaler=None,
    max_grad_norm: float = 1.0,
    ckpt_dir: str = "./checkpoints",
    early_stopping: bool = True,
    patience: int = 10,
    trial=None,          # [ADDED] Optuna trial 연결
    use_writer: bool = True,   # [ADDED] Optuna trial 중에는 writer 끄기 쉽게
):
    model.to(device)

    best_val_mae = float("inf")
    es = EarlyStopping(patience=patience, min_delta=1e-4) if early_stopping else None

    # [ADDED] SummaryWriter 생성
    writer = SummaryWriter()

    for epoch in range(1, num_epochs + 1):
        train_loss, train_mae, tsec = train_one_epoch(
            model=model,
            train_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            use_amp=use_amp,
            max_grad_norm=max_grad_norm,
            log_interval=50,
        )

        val_loss, val_mae = validate(
            model=model,
            val_loader=val_loader,
            criterion=criterion,
            device=device,
        )

        # Scheduler step
        if scheduler is not None:
            if scheduler_type.lower() == "plateau":
                scheduler.step(val_loss)  # 또는 val_mae로 step해도 됨
            else:
                scheduler.step()

        # Logging
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"\nEpoch {epoch}/{num_epochs} | "
            f"lr {current_lr:.2e} | "
            f"train loss {train_loss:.4f}, mae {train_mae:.4f} | "
            f"val loss {val_loss:.4f}, mae {val_mae:.4f} | "
            f"time {tsec:.1f}s"
        )

        # [ADDED] epoch 단위 TensorBoard 기록
        writer.add_scalar("Epoch/train_loss", train_loss, epoch)
        writer.add_scalar("Epoch/train_mae", train_mae, epoch)
        writer.add_scalar("Epoch/val_loss", val_loss, epoch)
        writer.add_scalar("Epoch/val_mae", val_mae, epoch)
        writer.add_scalar("Epoch/lr", current_lr, epoch)

        # Checkpoint (best val mae)
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_path = os.path.join(ckpt_dir, "best.pt")
            save_checkpoint(
                best_path, model, optimizer, epoch, best_val_mae,
                scheduler=scheduler, scaler=scaler
            )
            print(f"✅ Saved best checkpoint: {best_path} (best_val_mae={best_val_mae:.4f})")

        # Also save last (optional)
        last_path = os.path.join(ckpt_dir, "last.pt")
        save_checkpoint(
            last_path, model, optimizer, epoch, best_val_mae,
            scheduler=scheduler, scaler=scaler
        )

        # [ADDED] Optuna pruning/reporting
        if trial is not None:
            trial.report(val_mae, step=epoch)
            if trial.should_prune():
                if writer is not None:
                    writer.close()
                raise optuna.TrialPruned()

        # Early stopping
        if es is not None:
            should_stop = es.step(val_mae)  # MAE 기준으로 멈춤
            if should_stop:
                print(f"🛑 Early stopping triggered. Best val mae: {best_val_mae:.4f}")
                break

    # [ADDED] SummaryWriter 종료
    writer.close()

    return best_val_mae

In [ ]:
# [ADDED] 기존 loss builder 재사용 가능
def build_regression_loss(beta: float = 1.0) -> nn.Module:
    try:
        return nn.SmoothL1Loss(beta=beta)
    except TypeError:
        return nn.SmoothL1Loss()


# [ADDED] 기존 optimizer builder 재사용 가능
def build_adamw_optimizer(model: nn.Module, lr: float = 3e-4, weight_decay: float = 1e-2):
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.endswith(".bias") or ("norm" in name.lower()) or ("ln" in name.lower()):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    param_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    optimizer = optim.AdamW(param_groups, lr=lr, betas=(0.9, 0.999))
    return optimizer


# [ADDED] 기존 scheduler builder 재사용 가능
def build_scheduler(optimizer, scheduler_type: str, num_epochs: int):
    scheduler_type = scheduler_type.lower()

    if scheduler_type == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs, eta_min=1e-6
        )

    if scheduler_type == "plateau":
        return optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=5,
            threshold=1e-4,
            min_lr=1e-6,
        )

    raise ValueError(f"Unknown scheduler_type: {scheduler_type}")


# [ADDED] Optuna용 objective
def objective(
    trial,
    model_fn,              # model_fn() -> 새 모델 생성
    train_loader,
    val_loader,
    device,
    num_epochs=10,
    use_amp=True,
    ckpt_root="./optuna_checkpoints",
):
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 5e-2, log=True)
    beta = trial.suggest_float("smoothl1_beta", 0.3, 2.0)
    scheduler_type = trial.suggest_categorical("scheduler_type", ["cosine", "plateau"])
    max_grad_norm = trial.suggest_categorical("max_grad_norm", [0.5, 1.0, 2.0])

    # [ADDED] trial마다 모델 새로 생성
    model = model_fn().to(device)

    criterion = build_regression_loss(beta=beta)
    optimizer = build_adamw_optimizer(model, lr=lr, weight_decay=weight_decay)
    scheduler = build_scheduler(optimizer, scheduler_type, num_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type == "cuda")

    # [ADDED] trial별 checkpoint 경로 분리
    ckpt_dir = os.path.join(ckpt_root, f"trial_{trial.number}")

    best_val_mae = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        num_epochs=num_epochs,
        scheduler_type=scheduler_type,
        use_amp=(use_amp and device.type == "cuda"),
        scaler=scaler,
        max_grad_norm=max_grad_norm,
        ckpt_dir=ckpt_dir,
        early_stopping=True,
        patience=5,
        trial=trial,
        use_writer=False,   # [ADDED] trial마다 TensorBoard 로그 폭증 방지
    )

    return best_val_mae


# [ADDED] Optuna 실행 함수
def run_optuna(
    model_fn,
    train_loader,
    val_loader,
    device,
    n_trials=20,
    num_epochs=10,
    study_name="wheel_safe_optuna",
    storage="sqlite:///wheel_safe_optuna.db",
):
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    sampler = optuna.samplers.TPESampler(seed=42)

    study = optuna.create_study(
        direction="minimize",
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
        sampler=sampler,
        pruner=pruner,
    )

    study.optimize(
        lambda trial: objective(
            trial=trial,
            model_fn=model_fn,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            num_epochs=num_epochs,
            use_amp=True,
            ckpt_root="./optuna_checkpoints",
        ),
        n_trials=n_trials,
    )

    print("\n===== Optuna Best Trial =====")
    print("Best Value (val MAE):", study.best_value)
    print("Best Params:")
    for k, v in study.best_trial.params.items():
        print(f"  {k}: {v}")

    return study

Optimizer / Scheduler
- Optimizer: AdamW
- LR: pretrained fine-tuning이면 보통 1e-4 ~ 3e-4
- Weight decay: 1e-2 근처
- Scheduler:
  * 간단: CosineAnnealingLR
  * 또는 ReduceLROnPlateau(val_loss) (데이터 작을 때 안정)

### MAE (Mean Absolute Error)
> 평균 절대 오차
- 실제값(label)과 예측값(prediction)의 차이를 절대값으로 만든 뒤 평균을 낸 값
- 우리 프로젝트에서는 (MAE = 평균 경사각 오차 (deg))

HuberLoss는
- 작은 오차 → L2
- 큰 오차 → L1
이라서
MAE ≠ Loss
이다.

In [ ]:
def model_fn():
    model = ConvNeXtV2Regressor(
        backbone_name="convnextv2_tiny",
        pretrained=True,
        dropout=0.1
    )
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

study = run_optuna(
    model_fn=model_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    n_trials=20,
    num_epochs=20
)

[I 2026-03-09 19:43:32,232] Using an existing study with name 'wheel_safe_optuna' instead of creating a new one.
C:\Users\user\AppData\Local\Temp\ipykernel_45652\3723546259.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type == "cuda")
Train Epoch None:   0%|          | 0/19 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_45652\2815115808.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 1/20 | lr 4.30e-05 | train loss 4.2980, mae 5.0300 | val loss 0.7082, mae 1.2538 | time 7.2s
✅ Saved best checkpoint: ./optuna_checkpoints\trial_8\best.pt (best_val_mae=1.2538)



Epoch 2/20 | lr 4.22e-05 | train loss 0.8547, mae 1.3953 | val loss 0.7717, mae 1.3258 | time 7.1s



Epoch 3/20 | lr 4.10e-05 | train loss 3.4114, mae 4.0464 | val loss 5.8844, mae 6.6566 | time 7.2s



Epoch 4/20 | lr 3.92e-05 | train loss 2.8002, mae 3.5041 | val loss 0.8123, mae 1.3505 | time 7.2s


...

🛑 Early stopping triggered. Best val mae: 0.8316



Epoch 1/20 | lr 3.29e-05 | train loss 4.8820, mae 5.3900 | val loss 0.8881, mae 1.3099 | time 7.1s
✅ Saved best checkpoint: ./optuna_checkpoints\trial_10\best.pt (best_val_mae=1.3099)



Epoch 2/20 | lr 3.29e-05 | train loss 1.9496, mae 2.3980 | val loss 0.7136, mae 1.1237 | time 7.5s
✅ Saved best checkpoint: ./optuna_checkpoints\trial_10\best.pt (best_val_mae=1.1237)


Train Epoch None:   0%|          | 0/19 [00:02<?, ?it/s, loss=1.1423, mae=1.5480][W 2026-03-09 19:49:02,097] Trial 10 failed with parameters: {'lr': 3.2877474139911175e-05, 'weight_decay': 0.0008730885649333646, 'smoothl1_beta': 1.0343065316915967, 'scheduler_type': 'plateau', 'max_grad_norm': 2.0} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\wheel-safe\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_45652\3723546259.py", line 129, in <lambda>
    lambda trial: objective(
                  ^^^^^^^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_45652\3723546259.py", line 82, in objective
    best_val_mae = fit(
                   ^^^^
  File "C:\Users\user\AppData\Local\Temp\ipykernel_45652\2815115808.py", line 253, in fit
    train_loss, train_mae, tsec = train_one_epoch(
         

KeyboardInterrupt: 

In [ ]:
# model: ConvNeXtV2 regression 모델 (이미 생성했다고 가정)
# train_loader, val_loader: 이미 생성했다고 가정
# criterion, optimizer, scheduler, scaler, device, use_amp, MAX_GRAD_NORM: 이전 단계에서 준비

best_mae = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=NUM_EPOCHS,
    scheduler_type=SCHEDULER_TYPE,  # "cosine" or "plateau"
    use_amp=use_amp,
    scaler=scaler,
    max_grad_norm=MAX_GRAD_NORM,
    ckpt_dir="./checkpoints",
    early_stopping=True,
    patience=10,
)

print("Training done. Best val MAE:", best_mae)

C:\Users\user\AppData\Local\Temp\ipykernel_21912\223988614.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[Train] step 50/216 | loss 4.6341 | mae 5.0889
[Train] step 100/216 | loss 3.2785 | mae 3.7294
[Train] step 150/216 | loss 2.6247 | mae 3.0713
[Train] step 200/216 | loss 2.5812 | mae 3.0238

Epoch 1/50 | lr 3.00e-04 | train loss 2.4952, mae 2.9362 | val loss 2.6112, mae 3.0704 | time 101.6s
✅ Saved best checkpoint: ./checkpoints\best.pt (best_val_mae=3.0704)
[Train] step 50/216 | loss 1.7895 | mae 2.2406
[Train] step 100/216 | loss 2.0629 | mae 2.5222
[Train] step 150/216 | loss 2.1286 | mae 2.5728
[Train] step 200/216 | loss 2.3524 | mae 2.7960

Epoch 2/50 | lr 2.99e-04 | train loss 2.2646, mae 2.7072 | val loss 2.4138, mae 2.8702 | time 91.2s
✅ Saved best checkpoint: ./checkpoints\best.pt (best_val_mae=2.8702)
[Train] step 50/216 | loss 2.6856 | mae 3.1282
[Train] step 100/216 | loss 2.3797 | mae 2.8223
[Train] step 150/216 | loss 2.1173 | mae 2.5595
[Train] step 200/216 | loss 2.3449 | mae 2.7889

Epoch 3/50 | lr 2.97e-04 | train loss 2.2697, mae 2.7128 | val loss 2.3895, mae 2.839

In [ ]:
@torch.no_grad()
def evaluate_test(model, test_loader, device):
    
    model.eval()
    
    total_mae = 0
    n_samples = 0
    
    for imgs, labels in test_loader:
        
        imgs = imgs.to(device)
        labels = labels.to(device).view(-1)
        
        preds = model(imgs).view(-1)
        
        mae = torch.abs(preds - labels)
        
        total_mae += mae.sum().item()
        n_samples += imgs.size(0)
        
    test_mae = total_mae / n_samples
    
    return test_mae

In [ ]:
test_mae = evaluate_test(model, test_loader, device)

print("Test MAE:", test_mae)

Test MAE: 2.7785267742401962
